In [ ]:
##############################################
##############################################
### TO GET ACCESS TO FILE IN GOOGLE COLAB ###
##############################################
##############################################

# To get access to the data files from github in Google Colab
#!git clone https://github.com/math869c/graph_representation_st457.git

# set the folder to get access to the data
#import os
#os.chdir('/content') # to avoid error if rerun
#os.chdir('./graph_representation_st457')

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import torch.optim as optim
import json

# load from other folders
from helper_functions import *
from model_classes import *

2026-04-19 11:32:49.332302: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Load data for LSTM and TGC

In [3]:
# Stock prices
open_prices_interp = pd.read_csv('data_folder/open_prices_interp.csv', index_col=0)
# into numpy
x = open_prices_interp.to_numpy()
tickers_with_data = open_prices_interp.columns

# Load graph data
with open("data_folder/firm_industry.json", "r") as f:
    firm_industry_dict = json.load(f)
A = np.load("data_folder/adjacency_matrix.npy")
adj_matrix = torch.tensor(A, dtype=torch.float32)

# make training data
batch_size =32
X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(x, 
                                                                                                        batch_size =batch_size, 
                                                                                                        flatten_data = True, # Should be True for LSTM and GTC and False for GAT
                                                                                                        flatten_time_features=False # Should be False for LSTM and GTC and True for GAT
                                                                                                        )  

torch.Size([984, 8, 2300])
torch.Size([984, 460])


# Load LSTM model

In [ ]:
# this is j
output_size = y_train.shape[1]
input_size = X_train.shape[2]

model = LSTMRegressor(
    input_size=input_size,
    hidden_units=32,
    output_size=output_size,
    dropout_rate=0.2
)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters()) # just use standard learning rate for now


# Train LSTM Model

In [ ]:
mode_LSTM, history_metrics_LSTM, best_val_loss_LSTM = train_with_validation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=20,
    patience=10
)

test_loss, _ = evaluate(model_LSTM, test_loader, criterion)
print("Best validation loss:", best_val_loss)
print("Test loss:", test_loss)


# Load information for TGC model

In [ ]:
emb_dim = 32
K = adj_matrix.shape[-1]
x0, y0 = next(iter(train_loader))
B, L, N, F = x0.shape

# Train TGC model

In [ ]:
dict_adj_matrices = training_loop_TGC(train_loader=train_loader, val_loader=val_loader, y_test=y_test, adj_matrix=adj_matrix, F=F, emb_dim=emb_dim, K_num_relations =K, num_epochs = 20)

# Evaluate LSTM

In [ ]:
pred_scaled_LSTM = predict_model(model, test_loader_final)
# Manually inverse transform pred_scaled to undo the scaling
pred = pred_scaled_LSTM * sc.scale_[0] + sc.mean_[0]
real = y_test * sc.scale_[0] + sc.mean_[0]

In [ ]:
plot_results(y_test, real, open_prices_interp)

In [ ]:
average_metrics = compute_metrics(y_test, np.mean(y_train, axis=0)[np.newaxis, :])
lstm_metrics = compute_metrics(y_test, pred_scaled)

print(average_metrics)
print(lstm_metrics)

# Evaluate TGC Model

In [ ]:
MSE_dict = {'MSE_array_average': MSE_array_average, 'MSE_array_LSTM': MSE_array_LSTM}
for key in dict_adj_matrices.keys():
  MSE_dict[f'MSE_array_{key}'] = dict_adj_matrices[key]['MSE']